# Studi Kasus 4 — Klasifikasi Risiko Keterlambatan Layanan
**Materi:** Pengantar Machine Learning  
Workflow mencakup data, preprocessing, train/test split, model klasifikasi, evaluasi, dan keterbatasan.

## Rumusan masalah dan batasan
Model akan memprediksi label simulasi `berisiko_terlambat`. Dataset sintetis sengaja dibuat agar notebook dapat dijalankan tanpa unduhan eksternal. Hasil tidak untuk keputusan nyata dan tidak menunjukkan performa pada data operasional.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay,
                             roc_auc_score, accuracy_score)

rng = np.random.default_rng(2026)
n = 1000
distance = rng.uniform(1, 80, n)
workload = rng.integers(1, 12, n)
weather = rng.choice(["cerah", "hujan", "ekstrem"], n, p=[0.55, 0.35, 0.10])
priority = rng.choice(["normal", "tinggi"], n, p=[0.75, 0.25])
noise = rng.normal(0, 1.0, n)
logit = -3.0 + 0.035*distance + 0.20*workload + np.where(weather=="hujan", 0.7, 0) + np.where(weather=="ekstrem", 1.4, 0) - np.where(priority=="tinggi", 0.25, 0) + noise
prob = 1 / (1 + np.exp(-logit))
y = (rng.random(n) < prob).astype(int)
data = pd.DataFrame({"jarak_km": distance, "beban_tugas": workload,
                     "cuaca": weather, "prioritas": priority,
                     "berisiko_terlambat": y})
# Tambahkan beberapa nilai kosong agar pipeline menunjukkan imputasi.
data.loc[rng.choice(n, 20, replace=False), "jarak_km"] = np.nan
display(data.head())
print("Proporsi kelas:")
display(data["berisiko_terlambat"].value_counts(normalize=True).rename("proporsi"))

In [ ]:
X = data.drop(columns="berisiko_terlambat")
y = data["berisiko_terlambat"]
numeric = ["jarak_km", "beban_tugas"]
categorical = ["cuaca", "prioritas"]
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric),
    ("cat", categorical_pipe, categorical)
])
model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
model.fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print(classification_report(y_test, pred, target_names=["tidak berisiko", "berisiko"]))
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, pred, display_labels=["Tidak berisiko", "Berisiko"], cmap="Blues"
)
plt.title("Confusion matrix pada data uji")
plt.tight_layout()
plt.show()

## Interpretasi dan penggunaan bertanggung jawab
- Recall kelas berisiko penting bila melewatkan keterlambatan memiliki biaya tinggi; precision penting agar tim tidak mendapat terlalu banyak alarm palsu.
- `class_weight='balanced'` adalah pilihan eksperimen, bukan jaminan model adil.
- Model dilatih dan diuji pada data simulasi dari proses yang sama; data nyata dapat berbeda.
- Sebelum deployment, lakukan validasi temporal, analisis subgroup yang sah, kalibrasi, monitoring drift, keamanan, dan tinjauan manusia.